# 真实输入驱动的科研 Notebook 模板
用 `python input_adapter.py --init 数据.csv --config input_config.json` 生成配置草稿，补全来源、角色、字段映射、单位和缺失标记，再运行 `python input_adapter.py --config input_config.json` 预检。缺少真实输入会停止，不会回退模拟数据。
本模板以分组统计展示接入流程，实际论文复现须替换为已核验的方法及基线。真实观测示例见 real_data_example.ipynb。

## 1. 科学输入与可追溯配置

科学参数集中在此。分组标准差采用 ddof=1；单样本组的 SD 无定义，保留 NaN。所有未标记 `plot` 的代码单元都计入计算代码身份；绘图单元须带 `plot` 标签。

In [ ]:
from pathlib import Path
import platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd

from input_adapter import load_input, resolve_input
from notebook_provenance import (Checks, check_execution, local_code, require, science_hash,
                                 sha256, write_json)

INPUT_CONFIG = Path("input_config.json")
NOTEBOOK_FILE = Path("notebook_template.ipynb")  # 重命名 Notebook 后同步修改
PROJECT_ROOT = Path(".")  # 参与计算的本地模块所在根目录，导入的模块自动计入代码身份
OUTPUT_DIR = Path("example_output")
METHOD = "group-summary"
PARAMETERS = {"group_column": "group", "value_column": "value", "ddof": 1,
              "missing_policy": "exclude-empty-group-or-nonfinite-value"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = OUTPUT_DIR / "group_summary.csv"
OBSERVATIONS_CSV = OUTPUT_DIR / "observations_for_plot.csv"

input_rows, input_provenance = load_input(INPUT_CONFIG)  # 输入不满足约定时在此 BLOCKED
DATA_FILE = resolve_input(INPUT_CONFIG)
checks = Checks(OUTPUT_DIR / "validation_matrix.csv", dataset_id=input_provenance["dataset_id"], method=METHOD,
                refactor_code_version="notebook-sha256:" + science_hash(NOTEBOOK_FILE),
                expected=["input:availability", "input:identity", "input:schema", "input:units", "input:role",
                          "input:qc", "input:preview", "execution:source-match", "execution:clean-kernel"])
checks.check("input:availability", DATA_FILE.is_file(), "Input file must exist")
checks.check("input:identity", sha256(DATA_FILE) == input_provenance["sha256"], "Input hash must match the config")
checks.check("input:units", input_provenance["units"]["value"], "Value unit must be declared")
checks.check("input:role", input_provenance["role"], "Scientific role must be declared")

## 2. 输入检查与 QC

标识符按字符串读取；不把无效值变成零。逐条保存被排除记录及原因，数据不足时停止。

In [ ]:
raw = pd.DataFrame(input_rows)
print("Input source:", input_provenance["source"], "role:", input_provenance["role"])
required = {"sample_id", "sample_key", PARAMETERS["group_column"], PARAMETERS["value_column"]}
checks.check("input:schema", required <= set(raw.columns) and raw["sample_key"].is_unique
             and raw["sample_id"].str.strip().ne("").all(), "Required columns and unique nonempty keys")

# 逐条给出排除原因：声明的缺失、无法解析（如 <0.05、1,5）与非有限值不混为一类。
text = raw[PARAMETERS["value_column"]].str.strip()
number_pattern = r"[+-]?(?:[0-9]+\.?[0-9]*|\.[0-9]+)(?:[eE][+-]?[0-9]+)?"
is_number = text.str.fullmatch(number_pattern)
values = text.where(is_number, "nan").astype(float)
missing_group = raw[PARAMETERS["group_column"]].str.strip().eq("")
declared_missing = raw["value_raw"].isin(input_provenance["missing_values"])
nonfinite = ~declared_missing & (text.str.lower().isin(["nan", "inf", "+inf", "-inf", "infinity", "-infinity"])
                                 | (is_number & ~np.isfinite(values)))
undeclared_blank = ~declared_missing & text.eq("")
unparseable = ~declared_missing & ~nonfinite & ~undeclared_blank & ~is_number
reasons = {"empty_group": missing_group, "declared_missing": declared_missing, "undeclared_blank": undeclared_blank,
           "nonfinite_value": nonfinite, "unparseable_value": unparseable}
invalid = np.logical_or.reduce(list(reasons.values()))
excluded = raw.loc[invalid].copy()
excluded["qc_reason"] = [";".join(name for name, mask in reasons.items() if mask.loc[i]) for i in excluded.index]
excluded.to_csv(OUTPUT_DIR / "excluded_records.csv", index=False)
valid = raw.loc[~invalid].copy()
valid[PARAMETERS["value_column"]] = values.loc[valid.index]
qc_counts = {name: int(mask.sum()) for name, mask in reasons.items()}
checks.check("input:qc", len(valid) > 0 and len(valid) + len(excluded) == len(raw)
             and np.isfinite(valid[PARAMETERS["value_column"]]).all(), "Valid observations remain; every exclusion has a reason",
             rule=str(qc_counts))
print(f"Input={len(raw)}, valid={len(valid)}, excluded={len(excluded)}, reasons={qc_counts}")
if qc_counts["unparseable_value"] or qc_counts["undeclared_blank"]:
    print("REVIEW: undeclared blank or unparseable values were excluded; confirm against the data dictionary.")
if input_provenance["units_identified"] < len(raw):
    print("REVIEW: repeated measurements per sample_id; records are not independent replicates.")
display(valid.head(5))

## 3. 核心计算、保存与来源记录

示例仅演示分组均值、样本 SD，不代表任何领域算法验证。保存前核对实际执行的代码与已保存 Notebook 一致；未保存的修改或乱序运行会被拒绝，此时保存后重启 Kernel 并 Run All。

In [ ]:
require(PARAMETERS["ddof"] == 1, "This teaching implementation only supports sample SD with ddof=1")
# n 为有效记录数，n_units 为不同 sample_id 数；重复测量时两者不同，SD 描述记录间差异。
summary = valid.groupby(PARAMETERS["group_column"], sort=True).agg(
    n=(PARAMETERS["value_column"], "count"), n_units=("sample_id", "nunique"),
    mean=(PARAMETERS["value_column"], "mean"), sd=(PARAMETERS["value_column"], "std")
).reset_index().rename(columns={PARAMETERS["group_column"]: "group"})
require(int(summary["n"].sum()) == len(valid), "Group counts must cover all valid records")
require(np.isfinite(summary["mean"]).all(), "Group means must be finite")
require(summary.loc[summary["n"] == 1, "sd"].isna().all(), "SD of a single record is undefined")

# 保存前核对：实际执行的计算单元须与已保存的 Notebook 逐字一致，且按顺序运行。
execution = check_execution(NOTEBOOK_FILE)
for metric, key in (("execution:source-match", "source_match"), ("execution:clean-kernel", "clean_kernel")):
    if execution[key]:
        checks.set(metric, "PASS", rule="Executed cells compared with the saved notebook")
    else:
        checks.set(metric, "NOT-RUN", notes="Not verified; restart the kernel and Run All")
require(sha256(DATA_FILE) == input_provenance["sha256"], "Input changed during the run")

summary.to_csv(SUMMARY_CSV, index=False)
valid[["sample_id", "sample_key", "group", "value"]].to_csv(OBSERVATIONS_CSV, index=False)
calculation = {
    "schema_version": "1.1", "synthetic_example": False, "method": METHOD, "input": input_provenance,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "scientific_parameters": PARAMETERS, "science_code_sha256": science_hash(NOTEBOOK_FILE),
    "local_code_sha256": local_code(PROJECT_ROOT), "execution": execution,
    "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                    "platform": platform.platform()},
    "result": {"path": SUMMARY_CSV.name, "sha256": sha256(SUMMARY_CSV), "rows": len(summary),
               "observations": len(valid), "observations_path": OBSERVATIONS_CSV.name,
               "observations_sha256": sha256(OBSERVATIONS_CSV)},
    "qc": {"input_rows": len(raw), "excluded_rows": len(excluded), "reasons": qc_counts}
}
# 重新计算会清空旧图记录，避免旧图挂在新结果名下。
write_json(OUTPUT_DIR / "provenance.json", {"calculation": calculation, "figures": {}})
checks.set("input:preview", "NOT-RUN", notes="Review the displayed records, then record the result")
checks.save()
display(summary)
print("Example checks passed; scientific method validation still requires a real baseline.")

## 4. 独立绘图：干净 Kernel 只运行下一单元

输入路径、当前科学参数和样式在本单元明确可见；参数快照须与计算阶段一致。原始输入不可得时可重绘归档结果，但 `input:identity` 记为 BLOCKED。SD 无定义时不画误差条，不修改统计表中的 NaN。

In [ ]:
from pathlib import Path
import io
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image

from input_adapter import resolve_input
from notebook_provenance import (Checks, changed_code, notebook_sources, science_hash, sha256,
                                 sources_hash, write_json)

# ===== 绘图输入身份；更改科学参数需回到计算阶段显式重算 =====
INPUT_CONFIG = Path("input_config.json")
NOTEBOOK_FILE = Path("notebook_template.ipynb")
PROJECT_ROOT = Path(".")
OUTPUT_DIR = Path("example_output")
EXPECTED_PARAMETERS = {"group_column": "group", "value_column": "value", "ddof": 1,
                       "missing_policy": "exclude-empty-group-or-nonfinite-value"}
# ===== 只影响图形的参数 =====
WIDTH_MM, HEIGHT_MM = 180, 110  # 可修改试稿尺寸，并非通用期刊要求
FONT_SIZE = 10
DPI = 300
FONT_FAMILY = "DejaVu Sans"  # 中文标签可改为已安装的 Noto Sans CJK SC / PingFang SC
TITLE = "Group mean ± sample SD"
X_LABEL = None  # None 时使用原始字段名
Y_LABEL = None  # None 时使用原始字段名与登记单位
COLOR = "#4878A8"
FORMAT = "png"  # 每图一种格式：默认 png；可改 tif/tiff，目标规范明确要求矢量图时用 pdf/svg
if FORMAT not in ("png", "tif", "tiff", "pdf", "svg"):
    raise ValueError("Choose one format: png, tif, tiff, pdf or svg")
FIGURE_FILE = OUTPUT_DIR / f"group_mean_example.{FORMAT}"

# ===== 身份校验：状态由实际判断写入验收表，任何一项失败即停止 =====
manifest = json.loads((OUTPUT_DIR / "provenance.json").read_text(encoding="utf-8"))
calculation = manifest["calculation"]
result = calculation["result"]
result_path, observations_path = OUTPUT_DIR / result["path"], OUTPUT_DIR / result["observations_path"]
checks = Checks(OUTPUT_DIR / "validation_matrix.csv", dataset_id=calculation["input"]["dataset_id"],
                method=calculation["method"], refactor_code_version="notebook-sha256:" + science_hash(NOTEBOOK_FILE),
                expected=["provenance:result", "provenance:code", "provenance:parameters", "input:identity",
                          "figure:nonempty", "figure:valid-values", "figure:data-rows", "figure:no-glyph-warning",
                          "figure:within-canvas", "figure:visual-check"])
checks.check("provenance:result", sha256(result_path) == result["sha256"]
             and sha256(observations_path) == result["observations_sha256"], "Result changed; stop and investigate")
checks.check("provenance:parameters", EXPECTED_PARAMETERS == calculation["scientific_parameters"],
             "Science parameters changed; explicitly recompute")
changed = changed_code(calculation["local_code_sha256"], PROJECT_ROOT)
checks.check("provenance:code", science_hash(NOTEBOOK_FILE) == calculation["science_code_sha256"] and not changed,
             f"Science code changed; explicitly recompute {changed or ''}".strip())
# 输入能找到就必须一致；找不到时允许重绘归档结果，但写成 BLOCKED 而不是 PASS。
input_file = None
if INPUT_CONFIG.is_file():
    checks.check("input:identity", sha256(INPUT_CONFIG) == calculation["input"]["config_sha256"],
                 "Input config changed; explicitly recompute")
    input_file = resolve_input(INPUT_CONFIG)
input_verified = input_file is not None and input_file.is_file()
if input_verified:
    checks.check("input:identity", sha256(input_file) == calculation["input"]["sha256"],
                 "Input changed; explicitly recompute")
else:
    checks.set("input:identity", "BLOCKED", notes="Archived result: original input unavailable, not revalidated")
    print("BLOCKED input:identity - archived result; original input NOT revalidated.")

# ===== 读取保存结果 =====
# 关闭默认 NA 识别：名为 None、NA、null 的组是合法字符串。
plot_data = pd.read_csv(result_path, dtype=str, keep_default_na=False)
plot_data[["n", "n_units"]] = plot_data[["n", "n_units"]].astype(int)
plot_data["mean"] = plot_data["mean"].astype(float)
plot_data["sd"] = plot_data["sd"].replace("", "nan").astype(float)
observations = pd.read_csv(observations_path, dtype=str, keep_default_na=False)
observations["value"] = observations["value"].astype(float)
checks.check("figure:nonempty", len(plot_data) > 0 and len(observations) > 0, "No data to plot")
single = plot_data["n"] == 1
checks.check("figure:valid-values", np.isfinite(plot_data["mean"]).all() and np.isfinite(observations["value"]).all()
             and plot_data.loc[single, "sd"].isna().all() and np.isfinite(plot_data.loc[~single, "sd"]).all()
             and (plot_data.loc[~single, "sd"] >= 0).all(), "Plotted values must be finite; SD undefined only for n=1")
counts = observations["group"].value_counts()
checks.check("figure:data-rows", len(plot_data) == result["rows"] and len(observations) == result["observations"]
             and plot_data["group"].is_unique and set(counts.index) == set(plot_data["group"])
             and all(counts[row.group] == row.n for row in plot_data.itertuples()), "Unexpected plot row or group counts")

font_manager.findfont(font_manager.FontProperties(family=FONT_FAMILY), fallback_to_default=False)
x_label = X_LABEL or calculation["input"]["columns"]["group"]
y_label = Y_LABEL or f"{calculation['input']['columns']['value']} ({calculation['input']['units']['value']})"
with mpl.rc_context({"font.family": FONT_FAMILY, "font.size": FONT_SIZE, "savefig.bbox": None,
                     "text.usetex": False, "pdf.fonttype": 42}):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        fig, ax = plt.subplots(figsize=(WIDTH_MM / 25.4, HEIGHT_MM / 25.4), layout="constrained")
        x = np.arange(len(plot_data))
        # 显示真实观测的分布，横向偏移仅防止点重叠，不改变测量值。
        for index, group in enumerate(plot_data["group"]):
            values = observations.loc[observations["group"] == group, "value"]
            offset = np.linspace(-0.14, 0.14, len(values)) if len(values) > 1 else 0.0
            ax.scatter(np.full(len(values), index) + offset, values,
                       s=9, alpha=0.25, color=COLOR, linewidths=0)
        ax.scatter(x, plot_data["mean"], color="black", marker="D", s=24, zorder=4)
        has_sd = np.isfinite(plot_data["sd"])
        if has_sd.any():
            ax.errorbar(x[has_sd], plot_data.loc[has_sd, "mean"], yerr=plot_data.loc[has_sd, "sd"],
                        fmt="none", color="black", capsize=4)
        labels = [f"{row.group}\n(n={row.n}" + (f" records, {row.n_units} units" if row.n_units < row.n else "") + ")"
                  + ("\nSD undefined" if row.n == 1 else "") for row in plot_data.itertuples()]
        ax.set_xticks(x, labels, fontsize=FONT_SIZE)
        ax.set_xlabel(x_label, fontsize=FONT_SIZE)
        ax.set_ylabel(y_label, fontsize=FONT_SIZE)
        ax.set_title(TITLE, fontsize=FONT_SIZE + 1)
        fig.canvas.draw()
        box, (width, height) = fig.get_tightbbox(), fig.get_size_inches()
        overflow_mm = 25.4 * max(-box.x0, -box.y0, box.x1 - width, box.y1 - height)
        checks.check("figure:within-canvas", overflow_mm <= 0.1, "Labels exceed the canvas; enlarge it or shorten labels")
        # 先渲染到内存，检查通过后才写文件；TIFF 存为 RGB + LZW。
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png" if FORMAT in ("tif", "tiff") else FORMAT, dpi=DPI,
                    bbox_inches=None, facecolor="white")
    glyph = [str(item.message) for item in caught if "Glyph" in str(item.message) and "missing" in str(item.message)]
    checks.check("figure:no-glyph-warning", not glyph, f"Missing glyphs; configure an installed font {glyph[:3] or ''}".strip())
    if FORMAT in ("tif", "tiff"):
        with Image.open(buffer) as image:
            image.convert("RGB").save(FIGURE_FILE, format="TIFF", compression="tiff_lzw", dpi=(DPI, DPI))
    else:
        FIGURE_FILE.write_bytes(buffer.getvalue())
    # 其他警告不静默丢弃，供研究者判断。
    for item in caught:
        print(f"Render warning: {item.message}")
    plt.show()

manifest["figures"]["group_mean_example"] = {
    "file": FIGURE_FILE.name, "sha256": sha256(FIGURE_FILE), "result_sha256": result["sha256"],
    "input_revalidated": input_verified, "matplotlib": mpl.__version__,
    "plot_code_sha256": sources_hash(notebook_sources(NOTEBOOK_FILE)[1]),
    "style": {"width_mm": WIDTH_MM, "height_mm": HEIGHT_MM, "font_size": FONT_SIZE, "dpi": DPI,
              "font_family": FONT_FAMILY, "title": TITLE, "x_label": x_label, "y_label": y_label,
              "color": COLOR, "format": FORMAT}}
write_json(OUTPUT_DIR / "provenance.json", manifest)
checks.set("figure:visual-check", "NOT-RUN", notes=f"Open {FIGURE_FILE.name}: labels, overlap, legibility")
checks.save()

## 5. 交付边界

`example_output/` 包含统计表、绘图数据、QC 排除表、一份来源记录 `provenance.json`（计算与图件）、一份验收表 `validation_matrix.csv` 和单一格式图件。

各检查状态来自实际判断，不代表论文方法复现通过。`figure:visual-check` 与 `input:preview` 默认 NOT-RUN，实际查看后才可填写 PASS；重绘会将图件检查重置。真实项目还需原实现对照、单位与样本设计核验、方法来源和容差依据。